In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler
)

import seaborn as sns

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    confusion_matrix, 
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt


## ANALISIS DEL CSV

In [2]:
CSV_HOTEL_PATH = "../../data/raw/dataset.csv"
df_hotel = pd.read_csv(CSV_HOTEL_PATH)

In [ ]:
# Show data
df_hotel.info()

In [ ]:
df_hotel.dtypes

In [ ]:
df_hotel.shape


In [ ]:
#df_hotel.head(20).T
df_hotel.tail(10).T

In [ ]:
# df_hotel.describe(include="number").T
df_hotel['customer_type'].describe().T

In [ ]:
df_hotel['customer_type'].isna().sum()

In [ ]:
print(f"Country number: {df_hotel['country'].value_counts(normalize=True)}")
print(f"Null countries: {df_hotel['country'].isna().sum()}")
print(f"Meal number: {df_hotel['meal'].value_counts(normalize=True)}")

In [ ]:
precio_negativo = df_hotel[df_hotel['adr'] < 0]
precio_negativo.T

In [ ]:
df_hotel['is_canceled'].value_counts(normalize=True)

In [ ]:

num_df = df_hotel.select_dtypes(include=['number'])
numeric_columns = num_df.columns
print(f"Numeric columns: {numeric_columns}")
object_df = df_hotel.select_dtypes(include=['object'])
object_columns = object_df.columns
print(f"Object / Categorical columns: {object_columns}")

print(f"{df_hotel.isnull().sum()}")
df_hotel[df_hotel.duplicated(keep=False)]
cols_con_negativos = num_df.columns[(num_df < 0).any()].tolist()
print(f"Cols with negative values: {cols_con_negativos}")


In [ ]:
print(f"Number of categories by column:\n{df_hotel.select_dtypes(include='object').nunique().sort_values(ascending=False)}")
print("")
for i in range(len(object_columns)):
    if (df_hotel[object_columns[i]] == 'Undefined').sum() > 0:
        print(f"Count of {object_columns[i]} 'undefined' : {(df_hotel[object_columns[i]] == 'Undefined').sum()}")

## EDA

In [ ]:
num_cols = df_hotel.select_dtypes(include='number').columns.tolist()

plt.figure(figsize=(14, 10))
sns.heatmap(
    df_hotel[num_cols].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0
)
plt.title('Correlación entre variables numéricas')
plt.show()

In [ ]:
print(f"Paises total: {len(df_hotel['country'].unique())}")

frecuencias = df_hotel['country'].value_counts(normalize=True)
print(f"Paises con frecuencia <= 1%: {len(frecuencias[frecuencias <= 0.01].index.tolist())}")
print(f"Paises con frecuencia entre 1% y 5%: {len(frecuencias[frecuencias.between(0.01, 0.051)].index.tolist())}")
print(f"Paises con frecuencia entre 5% y 10%: {len(frecuencias[frecuencias.between(0.05, 0.1)].index.tolist())}")

print(f"Canales de distribución total: {df_hotel['distribution_channel'].value_counts()}")

frecuencias = df_hotel['distribution_channel'].value_counts(normalize=True)
print(f"Canales de distribución 'undefined': {len(df_hotel[df_hotel['distribution_channel'] == 'Undefined'].index.tolist())}")
print(f"Canales de distribución con frecuencia <= 1%: {frecuencias[frecuencias <= 0.01].index.tolist()}")
print(f"Canales de distribución con frecuencia entre 1% y 5%: {frecuencias[frecuencias.between(0.01, 0.051)].index.tolist()}")
print(f"Canales de distribución con frecuencia entre 5% y 10%: {frecuencias[frecuencias.between(0.05, 0.1)].index.tolist()}")
print(f"Canales de distribución con frecuencia >= 10%: {frecuencias[frecuencias >= 0.1].index.tolist()}")

stay_in_week_nights -> 0.5 -> stay_in_weekend_nights

## PREPROCESAMIENTO DE DATOS

In [3]:
df_hotel_prep = df_hotel.copy()

In [ ]:
# for i in range(len(object_columns)):
#     print(f"COLUMN {object_columns[i]}: {df_hotel_prep[object_columns[i]].unique()}")

print(f"{df_hotel_prep['distribution_channel'].value_counts()}")
print(f"{df_hotel_prep['meal'].value_counts()}")
print(f"{df_hotel_prep['market_segment'].value_counts()}")
print(f"{df_hotel_prep['country'].value_counts()}")
print(f"Count of reserved_room_type : {df_hotel_prep['reserved_room_type'].value_counts()}")
print(f"Count of assigned_room_type : {df_hotel_prep['assigned_room_type'].value_counts()}")
print(f"Count of adr price unique: {df_hotel_prep['adr'].unique()}")
print(f"Count of adr price == 1: {(df_hotel_prep['adr'] == 1).value_counts()}")
print(f"Count of adr price between 0 and 10: {(df_hotel_prep['adr'].between(0, 11)).value_counts()}")
print(f"Count of adr price < 0: {(df_hotel_prep['adr'] < 0).value_counts()}")

print(f"Count of distribution_channel: {(df_hotel_prep['distribution_channel']).sum()}")
print(f"Count of distribution_channel 'undefined' : {(df_hotel_prep['distribution_channel'] == 'Undefined').sum()}")
print(f"Count of meal 'undefined' : {(df_hotel_prep['meal'] == 'Undefined').sum()}")
print(f"Count of market_segment 'undefined' : {(df_hotel_prep['market_segment'] == 'Undefined').sum()}")
print(f"Count of country under 10% : {(df_hotel_prep['country'].value_counts(normalize=True) <= 0.1).sum()}")

In [4]:
# PREPROCESING
# children null -> lets say 0 childrens OK
# country null or < 10% -> Grupo 'others' (grupos = df.groupby(df['precio'] > 100)['precio'].count()) OK
# agent -> Value 0 if null value OK
# company null -> remove (+90% null) OK
# Remove duplicates OK

# Hotel -> LabelEncoder OK
# meal ->  OneHotEncoder (new meal type UM)
# country -> Grouping ( <= 10% (175) or > 10% (2)) + OneHotEncoder
# market_segment -> OneHotEncoder (remove undefined)
# distribution_channel -> OneHotEncoder (remove undefined)
# reserved_room_type and assigned_room_type -> OneHotEncoder
# deposit_type -> OneHotEncoder
# deposit_type -> OneHotEncoder
#columnas_drop = [
#    'arrival_date_year',          # Sobreajusta a años concretos
#    'arrival_date_week_number',   # Redundante con mes
#    'arrival_date_day_of_month',  # Ruido
#    'arrival_date_month',         # Ya codificada en sin/cos
#    'reservation_status_date',    # Es post-evento, data leakage
#]#
# Remove duplicates
print(f"Shape before removing duplicates: {df_hotel_prep.shape}")
print(f"Duplicated number before: {df_hotel_prep.duplicated(keep=False).sum()}")
df_hotel_prep = df_hotel_prep.drop_duplicates(keep='first').reset_index(drop=True)
print(f"Shape after removing duplicates: {df_hotel_prep.shape}")
print(f"Duplicated number after: {df_hotel_prep.duplicated(keep=False).sum()}")

Shape before removing duplicates: (119390, 32)
Duplicated number before: 40165
Shape after removing duplicates: (87396, 32)
Duplicated number after: 0


In [ ]:
df_hotel_prep.info()

In [5]:
df_hotel_prep['children'] = df_hotel_prep['children'].fillna(0)

In [28]:
df_hotel_prep.isna().sum()

hotel                             0
is_canceled                       0
lead_time                         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
booking_changes                   0
deposit_type                      0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces       0
total_of_special_requests         0
reserved_assigned_room            0
agent_known                       0
month_sin                         0
month_cos                         0
dtype: int64

In [6]:
# Borrado columnas
drop_columns = [
   'company',
   'arrival_date_year',          # Sobreajusta a años concretos
   'arrival_date_week_number',   # Redundante con mes
   'arrival_date_day_of_month',  # Ruido
   'reservation_status_date',    # Es post-evento, data leakage
   'reservation_status',         # Puede provocar overfitting
]
df_hotel_prep.drop(columns=drop_columns, inplace=True)

# Filas con precio negativo: outlier imposible que distorsiona el escalado
df_hotel_prep = df_hotel_prep[df_hotel_prep['adr'] >= 0].reset_index(drop=True)

# Reservas sin ningún huésped: datos incoherentes (children puede tener NaN aquí, tratado con fillna)
df_hotel_prep = df_hotel_prep[
    df_hotel_prep['adults'].fillna(0) + df_hotel_prep['children'].fillna(0) + df_hotel_prep['babies'].fillna(0) > 0
].reset_index(drop=True)

print(f"Shape tras limpieza de filas inválidas: {df_hotel_prep.shape}")

Shape tras limpieza de filas inválidas: (87229, 26)


In [7]:
df_hotel_prep.isna().sum()

hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_month                    0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              0
babies                                0
meal                                  0
country                             447
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
deposit_type                          0
agent                             12141
days_in_waiting_list                  0
customer_type                         0
adr                                   0
required_car_parking_spaces           0


In [8]:
# Agrupar los paises con frecuencia < 1%
frequencies = df_hotel_prep['country'].value_counts(normalize=True)
index_low_frequency = frequencies[frequencies <= 0.01].index
df_hotel_prep['country'] = np.where(
    df_hotel_prep['country'].isna() | df_hotel_prep['country'].isin(index_low_frequency),
    'Others',
    df_hotel_prep['country']
)
print(f"{df_hotel_prep['country'].value_counts(normalize=True)}")

country
PRT       0.313600
Others    0.119937
GBR       0.119490
FRA       0.101148
ESP       0.083046
DEU       0.061734
ITA       0.035092
IRL       0.034564
BEL       0.023857
BRA       0.022848
NLD       0.021896
USA       0.021449
CHE       0.017953
CN        0.012530
AUT       0.010856
Name: proportion, dtype: float64


In [9]:
# Agrupar los canales de distribución con frecuencia < 1% (Undefined -> news)
df_hotel_prep['distribution_channel'] = df_hotel_prep['distribution_channel'].replace('Undefined', 'news')

frequencies = df_hotel_prep['distribution_channel'].value_counts(normalize=True)
index_low_frequency = frequencies[frequencies < 0.01].index
df_hotel_prep['distribution_channel'] = np.where(
    df_hotel_prep['distribution_channel'].isna() | df_hotel_prep['distribution_channel'].isin(index_low_frequency),
    'others',
    df_hotel_prep['distribution_channel']
)
print(f"{df_hotel_prep['distribution_channel'].value_counts(normalize=True)}")

distribution_channel
TA/TO        0.791342
Direct       0.148494
Corporate    0.058031
others       0.002132
Name: proportion, dtype: float64


In [10]:
# Agrupar el segmento de mercado con frecuencia <= 1%
frequencies = df_hotel_prep['market_segment'].value_counts(normalize=True)
index_low_frequency = frequencies[frequencies <= 0.01].index
df_hotel_prep['market_segment'] = np.where(
    df_hotel_prep['market_segment'].isna() | df_hotel_prep['market_segment'].isin(index_low_frequency),
    'Others',
    df_hotel_prep['market_segment']
)
print(f"{df_hotel_prep['market_segment'].value_counts(normalize=True)}")

market_segment
Online TA        0.591008
Offline TA/TO    0.158835
Direct           0.135047
Groups           0.056415
Corporate        0.048149
Others           0.010547
Name: proportion, dtype: float64


In [9]:
# Nueva columna indicando si ambas coinciden
df_hotel_prep['reserved_assigned_room'] = (df_hotel_prep['reserved_room_type'] == df_hotel_prep['assigned_room_type']).astype(int)
df_hotel_prep['reserved_assigned_room']

# SOLO PARA REGRESION LOGISTICA O REDES NEURONALES
df_hotel_prep.drop(columns=['assigned_room_type'], inplace=True)

In [10]:
# 'Undefined' se mantiene como categoría propia: indica reserva sin plan especificado, no es lo mismo que 'NA'
df_hotel_prep['meal'].value_counts(normalize=True)

meal
BB           0.778480
SC           0.107659
HB           0.104094
Undefined    0.005640
FB           0.004127
Name: proportion, dtype: float64

In [11]:
df_hotel_prep['agent_known'] = df_hotel_prep['agent'].notna().astype(int)
df_hotel_prep.drop(columns=['agent'], inplace=True)
print(f"agent_known distribution:\n{df_hotel_prep['agent_known'].value_counts(normalize=True)}")

agent_known distribution:
agent_known
1    0.860815
0    0.139185
Name: proportion, dtype: float64



#### APLICAR METODOS PARA EVITAR SESGOS

In [12]:
df_hotel_prep['hotel'] = df_hotel_prep['hotel'].map({'Resort Hotel': 0, 'City Hotel': 1})

In [13]:
# lead_time tiene distribución muy sesgada a la derecha (cola hasta ~700 días);
# log1p linealiza la relación con la cancelación para la LR, que no puede capturarla sola
df_hotel_prep['lead_time_log'] = np.log1p(df_hotel_prep['lead_time'])
df_hotel_prep.drop(columns=['lead_time'], inplace=True)

In [14]:
MONTHS = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
          'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}
month_num = df_hotel_prep['arrival_date_month'].map(MONTHS)
df_hotel_prep['month_sin'] = np.sin(2 * np.pi * month_num / 12)
df_hotel_prep['month_cos'] = np.cos(2 * np.pi * month_num / 12)

In [15]:
df_hotel_prep.drop(columns=['arrival_date_month'], inplace=True)

In [16]:
df_hotel_prep['children'] = df_hotel_prep['children'].clip(upper=df_hotel_prep['children'].quantile(0.99))
print(f"Agrupacion de children (quartil 99): {df_hotel_prep['children'].value_counts(normalize=True)}")

df_hotel_prep['adults'] = df_hotel_prep['adults'].clip(upper=df_hotel_prep['adults'].quantile(0.99))
print(f"Agrupacion de adults (quartil 99): {df_hotel_prep['adults'].value_counts(normalize=True)}")

df_hotel_prep['babies'] = df_hotel_prep['babies'].clip(upper=df_hotel_prep['babies'].quantile(0.99))
print(f"Agrupacion de babies (quartil 99): {df_hotel_prep['babies'].value_counts(normalize=True)}")

df_hotel_prep['adr'] = df_hotel_prep['adr'].clip(upper=df_hotel_prep['adr'].quantile(0.99))
print(f"Agrupacion de adr (quartil 99): {df_hotel_prep['adr'].value_counts(normalize=True)}")

Agrupacion de children (quartil 99): children
0.0    0.904114
1.0    0.053824
2.0    0.042062
Name: proportion, dtype: float64
Agrupacion de adults (quartil 99): adults
2    0.739387
1    0.189192
3    0.068911
0    0.002511
Name: proportion, dtype: float64
Agrupacion de babies (quartil 99): babies
0    0.989522
1    0.010478
Name: proportion, dtype: float64
Agrupacion de adr (quartil 99): adr
0.0000      0.018835
75.0000     0.015133
65.0000     0.014445
48.0000     0.010065
261.6224    0.010008
              ...   
55.1100     0.000011
33.6500     0.000011
31.4100     0.000011
31.8000     0.000011
157.7100    0.000011
Name: proportion, Length: 8516, dtype: float64


In [17]:
df_hotel_prep['adr'] = df_hotel_prep['adr'].clip(upper=df_hotel_prep['adr'].quantile(0.99))
print(f"Agrupacion de adr (quartil 99): {df_hotel_prep['adr'].value_counts(normalize=True)}")

Agrupacion de adr (quartil 99): adr
0.000000      0.018835
75.000000     0.015133
65.000000     0.014445
48.000000     0.010065
261.588128    0.010008
                ...   
55.110000     0.000011
33.650000     0.000011
31.410000     0.000011
31.800000     0.000011
157.710000    0.000011
Name: proportion, Length: 8516, dtype: float64


#### EDA después del preprocesamiento

In [ ]:
# Muestra qué columnas están más relacionadas con que el cliente cancele
corr_cancelacion = df_hotel_prep.corr(numeric_only=True)['is_canceled'].sort_values(ascending=False)
# print(corr_cancelacion)
positive_corr_cols = corr_cancelacion[corr_cancelacion > 0].index.tolist()
negative_corr_cols = corr_cancelacion[corr_cancelacion < 0].index.tolist()
positive_corr_cols.remove('is_canceled')
print(f"Positive correlation columns: {positive_corr_cols}")
print(f"Negative correlation columns: {negative_corr_cols}")

#### Obtener variables X e y

In [46]:
random_state=11
target_column = 'is_canceled'
X = df_hotel_prep.drop(columns=[target_column])
y = df_hotel_prep[target_column]

print(f"Percentage y: {y.value_counts(normalize=True)}")

Percentage y: is_canceled
0    0.724759
1    0.275241
Name: proportion, dtype: float64


In [49]:
model_lr = LogisticRegression(C=0.1, max_iter=500, penalty='l2', solver='liblinear', random_state=random_state, class_weight='balanced')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state, stratify=y)


# OHE se aplica después del split (fit solo con train) para evitar leakage
category_columns = ['meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type',
                    'deposit_type', 'customer_type', 'reserved_room_type', 'assigned_room_type']
ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[category_columns])
train_ohe = pd.DataFrame(ohe.transform(X_train[category_columns]), columns=ohe.get_feature_names_out(category_columns), index=X_train.index)
test_ohe  = pd.DataFrame(ohe.transform(X_test[category_columns]),  columns=ohe.get_feature_names_out(category_columns), index=X_test.index)
X_train = pd.concat([X_train.drop(columns=category_columns), train_ohe], axis=1)
X_test  = pd.concat([X_test.drop(columns=category_columns),  test_ohe],  axis=1)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print(f"Train data percentage: {y_train.value_counts(normalize=True)}")
print(f"Test data percentage:  {y_test.value_counts(normalize=True)}")

model_lr = model_lr.fit(X_train_scaled, y_train)

Train data percentage: is_canceled
0    0.724761
1    0.275239
Name: proportion, dtype: float64
Test data percentage:  is_canceled
0    0.724751
1    0.275249
Name: proportion, dtype: float64


In [50]:
y_predict = model_lr.predict(X_test_scaled)
y_predict_proba = model_lr.predict_proba(X_test_scaled)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_predict)
prec = precision_score(y_test, y_predict)
rec = recall_score(y_test, y_predict)
f1 = f1_score(y_test, y_predict)

# Cálculo del AUC (Area Under the Curve)
auc = roc_auc_score(y_test, y_predict_proba[:, 1])

print(f"Accuracy (Cancelaciones sobre el total):  {acc:.2%}")
print(f"Precisión (Cancelaciones predichas acertadas): {prec:.2%}")
print(f"Recall (Cancelaciones predichas vs reales):    {rec:.2%}")
print(f"F1-Score (media entre precision y recall):  {f1:.2%}")
print(f"AUC:       {auc:.2%}\n")

Accuracy (Cancelaciones sobre el total):  76.34%
Precisión (Cancelaciones predichas acertadas): 54.71%
Recall (Cancelaciones predichas vs reales):    81.61%
F1-Score (media entre precision y recall):  65.51%
AUC:       85.71%



#### Prueba con GridSearchCV


In [ ]:
param_grid = {
    'max_iter': [250, 300, 400, 600, 800],
    'penalty': ['l1', 'l2'], # Lasso. No hay muchas columnas
    'C': [0.05, 0.1, 0.2, 0.5, 1.0],
    'solver': ['liblinear']
}
multimetrics = ['accuracy', 'f1', 'roc_auc', 'precision', 'recall']
refit = 'recall'
# Elijo recall porque quiero atrapar a todos los que pueden cancelar. no quiero falsos negativos
# Elijo Lasso por evitar multicolinealidad
model_gs = GridSearchCV (model_lr, cv=5, scoring=refit, verbose=1, param_grid=param_grid)
model_gs.fit(X_train_scaled, y_train)

print(f"Mejores hiperparámetros: {model_gs.best_params_}")
print(f"Mejor puntuación de validación cruzada: {model_gs.best_score_}")
best_model_lr = model_gs.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits


In [ ]:
y_predict = best_model_lr.predict(X_test_scaled)
y_predict_proba = best_model_lr.predict_proba(X_test_scaled)


In [ ]:
print(f"Clases modelo: {best_model_lr.classes_}")

In [ ]:
print(f"Primeras 5 etiquetas reales: {y_test.iloc[-5:].values}")
print(f"Primeras 5 probabilidades: {y_predict_proba[-5:]}")
print(f"Primeras 5 predicciones: {y_predict[-5:]}")

In [ ]:
# Obtención de las métricas de evaluación
acc = accuracy_score(y_test, y_predict)
prec = precision_score(y_test, y_predict)
rec = recall_score(y_test, y_predict)
f1 = f1_score(y_test, y_predict)

# Cálculo del AUC (Area Under the Curve)
auc = roc_auc_score(y_test, y_predict_proba[:, 1])

print(f"Accuracy:  {acc:.2%}")
print(f"Precisión: {prec:.2%}")
print(f"Recall:    {rec:.2f}")
print(f"F1-Score:  {f1:.2f}")
print(f"AUC:       {auc:.2f}\n")

In [ ]:
print("\nReporte de clasificación:")
print(classification_report(y_test, y_predict))

In [ ]:
cm = confusion_matrix(y_test, y_predict)

# 3. Mostrarla de forma visual y bonita
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model_lr.classes_)
disp.plot(cmap='Blues')

plt.title(f'Matriz de Confusión (Refit: {refit})')
plt.show()

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_predict_proba)
plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.show()